In [78]:
import pandas as pd
import os
import subprocess

In [79]:
# Get repo root and set folders
root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()
data_folder = os.path.join(root, "GBI Optimisation", "data")
output_folder = os.path.join(root, "GBI Optimisation")

# Get excel file and sheets
excel_path = os.path.join(data_folder, "salaryP2 Nominal Inflation Adjusted.xlsx")

# Load and prepare salary data
salary_data = pd.read_excel(excel_path)
salary_data.columns = ['Year', 'Monthly_Gross_Salary']

# Convert to yearly salary for calculations
salary_data['Annual_Gross_Salary'] = salary_data['Monthly_Gross_Salary'] * 12


In [80]:
# Define constants
PENSION_RATE = 0.05
AM_BIDRAG_RATE = 0.08
BOTTOM_STATE_TAX_RATE = 0.1215
MUNICIPAL_TAX_RATE = 0.25
TOP_STATE_TAX_RATE = 0.15
TOP_TAX_THRESHOLD = 618400  # 2025 threshold (DKK)

In [81]:
# Function to calculate annual post-tax salary
def calculate_annual_post_tax_salary(annual_gross_salary):
    salary_after_pension = annual_gross_salary * (1 - PENSION_RATE)
    salary_after_am = salary_after_pension * (1 - AM_BIDRAG_RATE)

    bottom_state_tax = salary_after_am * BOTTOM_STATE_TAX_RATE
    municipal_tax = salary_after_am * MUNICIPAL_TAX_RATE
    top_state_tax = max(salary_after_am - TOP_TAX_THRESHOLD, 0) * TOP_STATE_TAX_RATE

    total_tax = bottom_state_tax + municipal_tax + top_state_tax
    post_tax_salary = salary_after_am - total_tax

    return post_tax_salary


# Apply function
salary_data['Annual_Post_Tax_Salary'] = salary_data['Annual_Gross_Salary'].apply(calculate_annual_post_tax_salary)

# Convert back to monthly
salary_data['Monthly_Post_Tax_Salary'] = salary_data['Annual_Post_Tax_Salary'] / 12

# Save results
salary_data[['Year', 'Monthly_Gross_Salary', 'Monthly_Post_Tax_Salary']].to_csv(
    os.path.join(output_folder, "net_salaryP2.csv"), index=False
)

In [82]:
salary_data.to_csv(os.path.join(output_folder, "net_salaryP2.csv"))